In [1]:
import pandas as pd
def load_data(interference_file_name: str) -> pd.DataFrame:
    interference_df = pd.read_csv(interference_file_name, low_memory=False)

    # Encode info, channel columns to dummies
    columns_to_encode = ['info', 'channel']

    for col in columns_to_encode:
        interference_df = pd.get_dummies(interference_df, columns=[col], prefix='is')

    # Keep all relevant columns having an is_* prefix, drop the rest
    # use this for Samsung-trained models
    channels_to_keep = ['is_37', 'is_38', 'is_39', 'is_ADV_IND', 'is_ADV_NONCONN_IND', 'is_ADV_SCAN_IND', 'is_SCAN_REQ', 'is_adv_channel' ,'is_broadcast']
    # use this for Apple-trained models
    #channels_to_keep = ['is_37', 'is_38', 'is_39', 'is_ADV_IND', 'is_ADV_NONCONN_IND', 'is_SCAN_REQ', 'is_ADV_EXT_IND', 'is_ADV_SCAN_IND', 'is_AUX_ADV_IND', 'is_adv_channel' ,'is_broadcast']
    other_channel_cols = [col for col in interference_df.columns if col.startswith('is_') and col not in channels_to_keep]
    interference_df = interference_df.drop(columns=other_channel_cols)
    # add missing dummy channels
    missing_channels = [ch for ch in channels_to_keep if ch not in interference_df.columns]
    for ch in missing_channels:
        interference_df[ch] = False



    return interference_df

In [2]:
def combine_data(interference_df: pd.DataFrame, target_labels: list) -> pd.DataFrame:
    # Aggregate dataframe by source and calculate new columns
    interfered_df_agg = aggregate_features('source', interference_df)
    interfered_df_calc = calculate_features(interference_df, interfered_df_agg)

    interfered_df_calc['group_label'] = interfered_df_calc['labelled_device'].apply(
        lambda x: 1 if x in target_labels else 0
    )

    return interfered_df_calc

In [3]:
# Group dataframe by source and extract some metrics
def aggregate_features(group_filter: str, input_df: pd.DataFrame) -> pd.DataFrame:
    input_df = input_df.sort_values(by=['source', 'time'])
    input_df['time_diff'] = input_df.groupby('source')['time'].diff()

    agg_df = input_df.groupby(group_filter).agg(
        ad_type_mean_length=('ad_type', lambda x: x.str.len().mean()),
        data_mean_length=('data', lambda x: x.str.len().mean()),
        uuid16_mean_length=('uuid16', lambda x: x.astype(str).str.len().mean()), # CHANGED: all values are NAN, so x.str gives an error
        service_data_mean_length=('service_data', lambda x: x.astype(str).str.len().mean()),
        mean_time_between_packets=('time_diff', 'mean'),
    )

    return agg_df

In [4]:
# Calculate some source-based feature columns
def calculate_features(actual_df: pd.DataFrame, aggregated_df: pd.DataFrame) -> pd.DataFrame:
    actual_df = actual_df.sort_values(by=['source', 'time'])
    actual_df['time_to_next'] = actual_df.groupby('source')['time'].shift(-1) - actual_df['time']
    actual_df['time_to_next'] = actual_df['time_to_next'].fillna(0)
    actual_df = actual_df.sort_values(by=['source', 'time'])
    actual_df['time_diff'] = actual_df.groupby('source')['time'].diff()

    group_means = aggregated_df['mean_time_between_packets']
    actual_df['mean_time_between_packets'] = actual_df['source'].map(group_means)
    actual_df['mean_time_between_packets'] = actual_df['mean_time_between_packets'].fillna(0)

    sd_group_means = aggregated_df['service_data_mean_length']
    actual_df['service_data_mean_length'] = actual_df['source'].map(sd_group_means)
    actual_df['service_data_mean_length'] = actual_df['service_data_mean_length'].fillna(0)

    actual_df['packet_send_freq'] = actual_df['time_to_next'] / actual_df['mean_time_between_packets']
    actual_df['packet_send_freq'] = actual_df['packet_send_freq'].fillna(0)

    actual_df['service_data_len_ratio'] = actual_df['service_data'].str.len() / actual_df['service_data_mean_length']
    actual_df['service_data_len_ratio'] = actual_df['service_data_len_ratio'].fillna(0)

    ad_type_group_means = aggregated_df['ad_type_mean_length']
    actual_df['ad_type_mean_length'] = actual_df['source'].map(ad_type_group_means)
    actual_df['ad_type_mean_length'] = actual_df['ad_type_mean_length'].fillna(0)
    actual_df['ad_type_len_ratio'] = actual_df['ad_type'].str.len() / actual_df['ad_type_mean_length']
    actual_df['ad_type_len_ratio'] = actual_df['ad_type_len_ratio'].fillna(0)

    data_group_means = aggregated_df['data_mean_length']
    actual_df['data_mean_length'] = actual_df['source'].map(data_group_means)
    actual_df['data_mean_length'] = actual_df['data_mean_length'].fillna(0)
    actual_df['data_len_ratio'] = actual_df['data'].str.len() / actual_df['data_mean_length']
    actual_df['data_len_ratio'] = actual_df['data_len_ratio'].fillna(0)

    uuid16_group_means = aggregated_df['uuid16_mean_length']
    actual_df['uuid16_mean_length'] = actual_df['source'].map(uuid16_group_means)
    actual_df['uuid16_mean_length'] = actual_df['uuid16_mean_length'].fillna(0)
    actual_df['uuid16_len_ratio'] = actual_df['uuid16'].astype(str).str.len() / actual_df['uuid16_mean_length']
    actual_df['uuid16_len_ratio'] = actual_df['uuid16_len_ratio'].fillna(0)

    # Drop intermediate columns as they were only used to help the calculation
    actual_df = actual_df.drop(columns=['mean_time_between_packets', 'time_diff', 'service_data_mean_length', 'ad_type_mean_length', 'data_mean_length', 'uuid16_mean_length'])

    return actual_df

In [5]:
# Single-feature ablation without any is_* features - 10h apartment (2Groups_3Devices, Hours0-8 train)
# Mirrors last code block of AblationStudy.ipynb (b3f42f2063177504), binary Target vs Noise
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import GroupShuffleSplit
from bisect import bisect_left, bisect_right

target_list_10h = ['Phone', 'Buds', 'AirTag']

base_10h = '../../data/2Groups_3Devices_10h_apartment'
h02 = load_data(f'{base_10h}/2Groups_3Devices_10h_apartment_Hours0To2_labeled.csv')
h24 = load_data(f'{base_10h}/2Groups_3Devices_10h_apartment_Hours2To4_labeled.csv')
h46 = load_data(f'{base_10h}/2Groups_3Devices_10h_apartment_Hours4To6_labeled.csv')
h68 = load_data(f'{base_10h}/2Groups_3Devices_10h_apartment_Hours6To8_labeled.csv')

int_h02 = combine_data(h02, target_list_10h)
int_h24 = combine_data(h24, target_list_10h)
int_h46 = combine_data(h46, target_list_10h)
int_h68 = combine_data(h68, target_list_10h)

train_df = pd.concat([int_h02, int_h24, int_h46, int_h68], ignore_index=True)
print(f"Train Hours0-8: {len(train_df)} packets, {train_df['source'].nunique()} sources")
print(train_df['group_label'].value_counts().to_dict())

# --- CODE FIELD 2: per-source packets-within windows (same as modeling_10h_apartment) ---
def count_within_window(group, window, col_name):
    times = group['time'].values
    counts = []
    for t in times:
        left = bisect_left(times, t - window)
        right = bisect_right(times, t + window)
        counts.append(right - left - 1)
    group[col_name] = counts
    return group

train_df = train_df.sort_values(by=['source', 'time']).reset_index(drop=True)
for _c in ['packets_within_1s', 'packets_within_100ms', 'packets_within_25ms', 'packets_within_2ms']:
    train_df[_c] = 0
train_df['time'] = train_df['time'].astype(float)
train_df = train_df.groupby('source', group_keys=False).apply(count_within_window, window=1.0, col_name='packets_within_1s')
train_df = train_df.groupby('source', group_keys=False).apply(count_within_window, window=0.1, col_name='packets_within_100ms')
train_df = train_df.groupby('source', group_keys=False).apply(count_within_window, window=0.025, col_name='packets_within_25ms')
train_df = train_df.groupby('source', group_keys=False).apply(count_within_window, window=0.002, col_name='packets_within_2ms')

# --- CODE FIELD 2b: group-interconnect features (temporal co-activity, payload sync, instance timing) ---
def add_group_interconnect_features(df, win=0.5):
    df = df.sort_values('time').copy()
    df['time'] = df['time'].astype(float)
    df['time_bin_500ms'] = (df['time'] // win).astype(int)
    bin_size = df.groupby('time_bin_500ms').size().rename('bin_total_packets')
    bin_sources = df.groupby('time_bin_500ms')['source'].nunique().rename('bin_distinct_sources')
    df = df.join(bin_size, on='time_bin_500ms')
    df = df.join(bin_sources, on='time_bin_500ms')
    df['coactive_packets_in_bin'] = df['bin_total_packets'] - 1
    df['coactive_sources_in_bin'] = df['bin_distinct_sources'] - 1
    df['coactive_density'] = df['coactive_packets_in_bin'] / (df['bin_total_packets'].clip(lower=1))
    df['is_solo_bin'] = (df['coactive_sources_in_bin'] == 0).astype(int)
    df['data_prefix10'] = df['data'].astype(str).str[:10]
    df['service_data_prefix8'] = df['service_data'].astype(str).str[:8]
    df['payload_changed'] = (df.groupby('source')['data_prefix10'].shift(1) != df['data_prefix10']).astype(int)
    df['payload_changed'] = df['payload_changed'].fillna(0)
    bin_flips = df.groupby('time_bin_500ms')['payload_changed'].sum().rename('bin_sync_flips')
    df = df.join(bin_flips, on='time_bin_500ms')
    df['payload_sync_in_bin'] = df['bin_sync_flips']
    df['payload_sync_density'] = df['bin_sync_flips'] / df['bin_distinct_sources'].clip(lower=1)
    df['time_diff'] = df.groupby('source')['time'].diff()
    iat_median = df.groupby('source')['time_diff'].median().rename('iat_median')
    iat_std = df.groupby('source')['time_diff'].std().rename('iat_std')
    iat_mean = df.groupby('source')['time_diff'].mean().rename('iat_mean')
    df = df.join(iat_median, on='source')
    df = df.join(iat_std, on='source')
    df = df.join(iat_mean, on='source')
    df['iat_std'] = df['iat_std'].fillna(0)
    df['iat_median'] = df['iat_median'].fillna(0)
    df['iat_mean'] = df['iat_mean'].fillna(0)
    df['iat_deviation'] = (df['time_diff'] - df['iat_median']).abs().fillna(0)
    df['is_burst_iat'] = (df['time_diff'] < 0.005).astype(int)
    df = df.drop(columns=['time_diff'])
    return df

train_df = add_group_interconnect_features(train_df, win=0.5)
print('Interconnect features added:', [c for c in ['coactive_packets_in_bin','payload_sync_in_bin','iat_median'] if c in train_df.columns])

try:
    dropped_columns
except NameError:
    dropped_columns = [
        'source', 'labelled_device', 'sublabel_device', 'time', 'ad_type', 'data', 'service_data',
        'uuid16', 'uuid128', 'manufacturer_data', 'rssi',
        'group_id', 'group_label_bin', 'group_label_multi',
        'time_bin_500ms', 'data_prefix10', 'service_data_prefix8',
    ]

base_X = train_df.drop(columns=[c for c in dropped_columns if c in train_df.columns] + ['group_label'], errors='ignore')
base_X = base_X.select_dtypes(include=['number', 'bool']).copy()
non_is_base_X = base_X[[c for c in base_X.columns if not c.startswith('is_')]].copy()
y = train_df['group_label'].astype(int)
groups = train_df['source'] if 'source' in train_df.columns else pd.Series(np.arange(len(train_df)))

feature_groups = {
    'length': [c for c in non_is_base_X.columns if c == 'length'],
    'has_company_id': [c for c in non_is_base_X.columns if c == 'has_company_id'],
    'has_uuid16': [c for c in non_is_base_X.columns if c == 'has_uuid16'],
    'len_uuid16': [c for c in non_is_base_X.columns if c == 'len_uuid16'],
    'has_uuid128': [c for c in non_is_base_X.columns if c == 'has_uuid128'],
    'len_data': [c for c in non_is_base_X.columns if c == 'len_data'],
    'len_ad_type': [c for c in non_is_base_X.columns if c == 'len_ad_type'],
    'len_service_data': [c for c in non_is_base_X.columns if c == 'len_service_data'],
    'time_start_end': [c for c in non_is_base_X.columns if c == 'time_start_end'],
    'delta_end_start': [c for c in non_is_base_X.columns if c == 'delta_end_start'],
    'delta_start_start': [c for c in non_is_base_X.columns if c == 'delta_start_start'],
    'time_to_next': [c for c in non_is_base_X.columns if c == 'time_to_next'],
    'packet_send_freq': [c for c in non_is_base_X.columns if c == 'packet_send_freq'],
    'service_data_len_ratio': [c for c in non_is_base_X.columns if c == 'service_data_len_ratio'],
    'ad_type_len_ratio': [c for c in non_is_base_X.columns if c == 'ad_type_len_ratio'],
    'data_len_ratio': [c for c in non_is_base_X.columns if c == 'data_len_ratio'],
    'uuid16_len_ratio': [c for c in non_is_base_X.columns if c == 'uuid16_len_ratio'],
    'burst_density_packets_within': [c for c in non_is_base_X.columns if c.startswith('packets_within_')],
    'temporal_coactivity': [c for c in non_is_base_X.columns if c in ['bin_total_packets', 'bin_distinct_sources', 'coactive_packets_in_bin', 'coactive_sources_in_bin', 'coactive_density', 'is_solo_bin']],
    'coordinated_payload_sync': [c for c in non_is_base_X.columns if c in ['payload_changed', 'bin_sync_flips', 'payload_sync_in_bin', 'payload_sync_density']],
    'instance_timing_fingerprint': [c for c in non_is_base_X.columns if c in ['iat_median', 'iat_std', 'iat_mean', 'iat_deviation', 'is_burst_iat']],
}
feature_groups = {k: v for k, v in feature_groups.items() if len(v) > 0}

non_is_features = [c for c in non_is_base_X.columns if c not in feature_groups]
print(f'Total features without is_*: {len(non_is_base_X.columns)}')
print(f'Single-feature ablation count: {len(feature_groups)}')
print(f'Kept as baseline-only extras: {len(non_is_features)}')

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(non_is_base_X, y, groups=groups))
X_train_full = non_is_base_X.iloc[train_idx]
X_test_full = non_is_base_X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

def train_eval(X_train, X_test, y_train, y_test):
    clf = RandomForestClassifier(
        random_state=1,
        class_weight='balanced',
        n_estimators=50,
        min_samples_split=10,
        min_samples_leaf=3,
        min_weight_fraction_leaf=0.002,
    )
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    p, r, f1, _ = precision_recall_fscore_support(y_test, pred, average='binary', zero_division=0)
    return {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1}

rows = []
baseline_metrics = train_eval(X_train_full, X_test_full, y_train, y_test)
rows.append({'setting': 'baseline_no_is_features', 'removed_feature': '', 'n_features': X_train_full.shape[1], **baseline_metrics})

for feat, cols_to_remove in feature_groups.items():
    keep_cols = [c for c in non_is_base_X.columns if c not in cols_to_remove]
    m = train_eval(X_train_full[keep_cols], X_test_full[keep_cols], y_train, y_test)
    rows.append({'setting': f'drop_{feat}', 'removed_feature': feat, 'n_features': len(keep_cols), **m})
    print('This feature is done: ', feat)

ablation_df = pd.DataFrame(rows)
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    ablation_df[f'delta_{metric}_vs_baseline'] = ablation_df[metric] - baseline_metrics[metric]
ablation_df = ablation_df.sort_values('delta_f1_vs_baseline').reset_index(drop=True)
display(ablation_df)
ablation_df.to_csv('10h_apartment_single_feature_ablation_no_is.csv', index=False)
print('Saved: 10h_apartment_single_feature_ablation_no_is.csv')


Train Hours0-8: 1842202 packets, 32951 sources
{0: 1108863, 1: 733339}


C:\Users\thoma\AppData\Local\Temp\ipykernel_2016\20082854.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_df = train_df.groupby('source', group_keys=False).apply(count_within_window, window=1.0, col_name='packets_within_1s')
C:\Users\thoma\AppData\Local\Temp\ipykernel_2016\20082854.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_df = train_df.groupby('source', group_keys=False).apply(count_withi

Interconnect features added: ['coactive_packets_in_bin', 'payload_sync_in_bin', 'iat_median']
Total features without is_*: 34
Single-feature ablation count: 21
Kept as baseline-only extras: 17
This feature is done:  length
This feature is done:  has_company_id
This feature is done:  has_uuid16
This feature is done:  len_uuid16
This feature is done:  has_uuid128
This feature is done:  len_data
This feature is done:  len_ad_type
This feature is done:  len_service_data
This feature is done:  time_start_end
This feature is done:  delta_end_start
This feature is done:  delta_start_start
This feature is done:  time_to_next
This feature is done:  packet_send_freq
This feature is done:  service_data_len_ratio
This feature is done:  ad_type_len_ratio
This feature is done:  data_len_ratio
This feature is done:  uuid16_len_ratio
This feature is done:  burst_density_packets_within
This feature is done:  temporal_coactivity
This feature is done:  coordinated_payload_sync
This feature is done:  inst

,setting,removed_feature,n_features,accuracy,precision,recall,f1,delta_accuracy_vs_baseline,delta_precision_vs_baseline,delta_recall_vs_baseline,delta_f1_vs_baseline
0,drop_instance_timing_fingerprint,instance_timing_fingerprint,30,0.996407,0.972825,0.984617,0.978685,-0.001377,-0.018718,0.002689,-0.008027
1,drop_has_company_id,has_company_id,33,0.997231,0.987641,0.979195,0.983400,-0.000554,-0.003903,-0.002733,-0.003312
2,drop_len_data,len_data,33,0.997260,0.989691,0.977476,0.983545,-0.000524,-0.001852,-0.004452,-0.003167
3,drop_len_ad_type,len_ad_type,33,0.997260,0.989691,0.977476,0.983545,-0.000524,-0.001852,-0.004452,-0.003167
4,drop_uuid16_len_ratio,uuid16_len_ratio,33,0.997408,0.989404,0.979548,0.984451,-0.000377,-0.002139,-0.002380,-0.002261
5,drop_service_data_len_ratio,service_data_len_ratio,33,0.997415,0.991022,0.978005,0.984471,-0.000369,-0.000521,-0.003923,-0.002241
6,drop_has_uuid128,has_uuid128,33,0.997438,0.988929,0.980385,0.984639,-0.000347,-0.002614,-0.001543,-0.002074
7,drop_len_service_data,len_service_data,33,0.997441,0.990237,0.979107,0.984641,-0.000343,-0.001306,-0.002821,-0.002072
8,drop_has_uuid16,has_uuid16,33,0.997449,0.988887,0.980562,0.984707,-0.000336,-0.002656,-0.001366,-0.002006
9,drop_coordinated_payload_sync,coordinated_payload_sync,30,0.997449,0.987500,0.981972,0.984728,-0.000336,-0.004043,0.000044,-0.001984


Saved: 10h_apartment_single_feature_ablation_no_is.csv
